In [9]:
# Basic Libraries
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Clustering
from sklearn.cluster import KMeans

# Evaluation
from sklearn.metrics import silhouette_score

In [10]:

# Load the data
df = pd.read_csv("/content/drive/MyDrive/MachineLearningModels/Unsupervised Learning/data.csv", encoding='latin1')")

# First 5 rows
display(df.head())

# Number of rows and columns
print("Shape:", df.shape)

# Column information
df.info()

# Statistical summary
display(df.describe())

# Missing values
display(df.isnull().sum())

SyntaxError: unterminated string literal (detected at line 2) (2002924772.py, line 2)

What we found

Our dataset has:

541,909 rows
8 columns
Description has 1,454 missing values
CustomerID has 135,080 missing values
InvoiceDate is currently stored as text

Data Preprocessing

Now we clean the data before doing business analysis.
Step 1 — Convert date

In [11]:
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

NameError: name 'df' is not defined

Step 2 — Handle missing descriptions
As We don't want missing product names.

In [ ]:
df["Description"] = df["Description"].fillna("Unknown")
display(df.isnull().sum())

Step 3 — Identify cancelled orders

Some invoices start with "C".

In [12]:
df["IsCancelled"] = df["InvoiceNo"].astype(str).str.startswith("C")
display(df[df['IsCancelled']].head(10))

print("Cancelled transactions:", df["IsCancelled"].sum())

NameError: name 'df' is not defined

In [ ]:
#Remove invalid sales transactions

#For revenue analysis, we only want actual positive sales.

sales_df = df[
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
].copy()
#This removes transactions with:negative quantity,zero quantity,zero/negative price
display(sales_df.head())
print("Shape:", sales_df.shape)


In [ ]:
#Step 5 — Create Revenue

#This is one of the most important calculations.

sales_df["Revenue"] = (
    sales_df["Quantity"] *
    sales_df["UnitPrice"]
)

##The formula is:

#Revenue = Quantity × Unit Price

In [ ]:
#calculate the important business numbers.
total_revenue = sales_df["Revenue"].sum()

total_orders = sales_df["InvoiceNo"].nunique()

total_customers = sales_df["CustomerID"].nunique()

average_order_value = (
    sales_df.groupby("InvoiceNo")["Revenue"].sum().mean()
)

print("Total Revenue: $", round(total_revenue, 2))
print("Total Orders:", total_orders)
print("Unique Customers:", total_customers)
print("Average Order Value: $", round(average_order_value, 2))


Graph 1 — Monthly Revenue Trend

In [ ]:
sales_df["YearMonth"] = (
    sales_df["InvoiceDate"]
    .dt.to_period("M")
    .astype(str)
)

monthly_revenue = (
    sales_df.groupby("YearMonth")["Revenue"]
    .sum()
)

plt.figure(figsize=(10, 5))

plt.plot(
    monthly_revenue.index,
    monthly_revenue.values,
    marker="o"
)

plt.title("Monthly Revenue Trend")
plt.xlabel("Month")
plt.ylabel("Revenue ($)")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

Revenue becomes much stronger toward the end of 2011, with November being the strongest month.

This immediately tells management that the business has significant seasonal growth toward the holiday period.

In [13]:
#Graph 2 — Top 10 Countries by Revenue

top_countries = (
    sales_df.groupby("Country")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
display(top_countries)

plt.figure(figsize=(10, 6))

plt.barh(
    top_countries.index,
    top_countries.values
)

plt.title("Top 10 Countries by Revenue")
plt.xlabel("Revenue ($)")
plt.ylabel("Country")

plt.tight_layout()
plt.show()

NameError: name 'sales_df' is not defined

The United Kingdom dominates revenue, generating approximately $9.03M of the $10.67M total.

That means the company is highly dependent on its UK market.



In [ ]:
#Graph 3 — Top 10 Products by Revenue
top_products = (
    sales_df.groupby("Description")["Revenue"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .sort_values()
)
display(top_products)

plt.figure(figsize=(10, 6))

plt.barh(
    top_products.index,
    top_products.values
)

plt.title("Top 10 Products by Revenue")
plt.xlabel("Revenue ($)")
plt.ylabel("Product")

plt.tight_layout()
plt.show()

The highest-revenue products include:

DOTCOM POSTAGE
REGENCY CAKESTAND 3 TIER
PAPER CRAFT, LITTLE BIRDIE
WHITE HANGING HEART T-LIGHT HOLDER
PARTY BUNTING

This helps management identify products that contribute strongly to revenue.

In [ ]:
#Graph 4 — Monthly Number of Orders
monthly_orders = (
    sales_df.groupby("YearMonth")["InvoiceNo"]
    .nunique()
)
display(monthly_orders)
plt.figure(figsize=(10, 5))

plt.plot(
    monthly_orders.index,
    monthly_orders.values,
    marker="o"
)

plt.title("Monthly Number of Orders")
plt.xlabel("Month")
plt.ylabel("Number of Orders")
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()

The number of orders rises significantly during September–November, with November having the highest order volume.

This supports the revenue trend and suggests a strong seasonal demand pattern.

In [14]:
# Set the latest date in the dataset as the reference date
snapshot_date = sales_df["InvoiceDate"].max() + pd.Timedelta(days=1)

# Calculate Recency, Frequency, and Monetary for each customer
rfm = sales_df.groupby("CustomerID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("InvoiceNo", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()
print(rfm)

# Display the first 5 customers
display(rfm.head())

NameError: name 'sales_df' is not defined

🔹 RFM Feature Engineering (Detailed Notes)

Recency, Frequency, and Monetary (RFM) is a behavioral segmentation technique used to quantify customer purchasing patterns.

* Recency measures how recently a customer made a purchase. Lower values indicate more recent activity.

* Frequency counts how many unique transactions a customer made.

* Monetary represents total spending by the customer.

By aggregating transactional data into customer-level metrics, we transform raw invoice data into structured behavioral features suitable for clustering.

Customers with negative monetary values (refunds) were removed to prevent distortion in cluster formation.